In [0]:
import json

REGISTRY = "cosmos_poc.ops.ingestion_sources"
REQUIRED = ["source_id", "cosmos_endpoint", "tenant_id", "subscription_id", "resource_group",
            "cosmos_database", "cosmos_container", "secret_scope", "client_id_secret", "client_secret_secret"]

rows = spark.sql(f"""
  SELECT source_id, cosmos_endpoint, tenant_id, subscription_id, resource_group,
         cosmos_database, cosmos_container, secret_scope, client_id_secret, client_secret_secret,
         coalesce(start_from, 'Beginning')                    AS start_from,
         cast(coalesce(max_items_per_batch, 100000) AS STRING) AS max_items_per_batch
  FROM {REGISTRY}
  WHERE enabled
  ORDER BY source_id""").collect()

sources = [{k: ("" if v is None else str(v)) for k, v in r.asDict().items()} for r in rows]

# --- validation: fail fast with a clear message instead of failing deep inside a For Each iteration
if not sources:
    raise ValueError(f"No enabled sources in {REGISTRY}")
bad = [(s["source_id"], [k for k in REQUIRED if not s[k] or s[k].startswith("<")]) for s in sources]
bad = [b for b in bad if b[1]]
if bad:
    raise ValueError(f"Registry rows with missing/placeholder values: {bad}")
keys = [(s["cosmos_endpoint"], s["cosmos_database"], s["cosmos_container"]) for s in sources]
if len(keys) != len(set(keys)):
    raise ValueError("Duplicate (endpoint, database, container) in registry - would collide on checkpoints/tables")

dbutils.jobs.taskValues.set(key="sources", value=sources)   # consumed by the For Each task
print(f"{len(sources)} enabled source(s):", [s["source_id"] for s in sources])